In this notebook we perform
1. Feature Engineering
2. Feature Selection
3. Modelling

In [ ]:
!pip install xgboost

In [ ]:
!pip install catboost

In [ ]:
#basic imports
import sys
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
# Display all the columns of the dataframe
pd.pandas.set_option('display.max_columns',None)
%matplotlib inline
mpl.style.use('ggplot')
sns.set_style('white')
sns.set_palette('viridis')
mpl.rcParams['figure.figsize'] = 6,4


# Modelling imports
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc


#Common Model Algorithms
from sklearn import svm, tree, linear_model, neighbors, naive_bayes, ensemble, discriminant_analysis, gaussian_process
from xgboost import XGBClassifier
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor


#ignore warnings
import warnings
warnings.filterwarnings('ignore')
print('-'*25)


In [ ]:
df = pd.read_csv('/Users/parthrajauria/Documents/MYPersonalProjects/EndToEndMLOPsProject/notebook/data/Student_performance_data _.csv')
df

In [ ]:
# Target Variable would be GPA
df.drop(['GradeClass','StudentID'], axis=1, inplace=True)
df.head()

In [ ]:
df.shape


# 5.1 LOG/FUNCTION TRANSFORMATIONS NEEDED?


In [ ]:
data = df.copy()


In [ ]:
# Getting this information from EDA notebook.
continuous_feature = ['StudyTimeWeekly', 'Absences', 'GPA']


In [ ]:
for feature in continuous_feature:
    data1 = df.copy()
    data1[feature] = np.log1p(data1[feature])
    #stats.probplot(df[feature], dist = "norm", plot = plt)
    stats.probplot(data1[feature], dist = "norm", plot = plt)
    plt.xlabel(feature)
    plt.ylabel("Count")
    plt.title(feature + 'Log Transformed')
    plt.show()

After applying the Log transformations, the distributions doesn't normalize, rather they deviate more. Therefore, we will not apply Log or any other Function transformations

# MODEL BUILDING

In [ ]:
df.columns


In [ ]:
X = df.drop('GPA', axis = 1)
y = df['GPA']

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape, X_test.shape


### FEATURE SCALING USING COLUMN_TRANSFORMER

In [ ]:

sc = StandardScaler()


In [ ]:
preprocessor = ColumnTransformer([
    ('sscaler', sc, X.columns),
])

In [ ]:
X_train_scaled = preprocessor.fit_transform(X_train)
X_train_scaled.shape

In [ ]:
X_test_scaled = preprocessor.transform(X_test)
X_test_scaled.shape

# MODEL TRAINING

In [ ]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square



In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}
model_list = []
r2_list =[]

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train_scaled, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 0.1962
- Mean Absolute Error: 0.1592
- R2 Score: 0.9541

Model performance for Test set
- Root Mean Squared Error: 0.1966
- Mean Absolute Error: 0.1553
- R2 Score: 0.9532

## ===================================


Lasso
Model performance for Training set
- Root Mean Squared Error: 0.9163
- Mean Absolute Error: 0.7736
- R2 Score: 0.0000

Model performance for Test set
- Root Mean Squared Error: 0.9097
- Mean Absolute Error: 0.7660
- R2 Score: -0.0008

## ===================================


Ridge
Model performance for Training set
- Root Mean Squared Error: 0.1962
- Mean Absolute Error: 0.1592
- R2 Score: 0.9541

Model performance for Test set
- Root Mean Squared Error: 0.1967
- Mean Absolute Error: 0.1553
- R2 Score: 0.9532
## ===================================


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 0.3600
- Mean Absolute Error: 0.2908
- R2 Score: 0.8456

Model performance for Test set
- Root Mean Squared Error: 0.4615
- Mean Absolute Error: 0.3659
- R2 Score: 0.7425
## ===================================


Decision Tree
Model performance for Training set
- Root Mean Squared Error: 0.0000
- Mean Absolute Error: 0.0000
- R2 Score: 1.0000

Model performance for Test set
- Root Mean Squared Error: 0.3392
- Mean Absolute Error: 0.2676
- R2 Score: 0.8609
## ===================================


Random Forest Regressor
Model performance for Training set
- Root Mean Squared Error: 0.0903
- Mean Absolute Error: 0.0715
- R2 Score: 0.9903

Model performance for Test set
- Root Mean Squared Error: 0.2460
- Mean Absolute Error: 0.1907
- R2 Score: 0.9268
## ===================================


XGBRegressor
Model performance for Training set
- Root Mean Squared Error: 0.0516
- Mean Absolute Error: 0.0371
- R2 Score: 0.9968

Model performance for Test set
- Root Mean Squared Error: 0.2398
- Mean Absolute Error: 0.1891
- R2 Score: 0.9305
## ===================================


CatBoosting Regressor
Model performance for Training set
- Root Mean Squared Error: 0.1167
- Mean Absolute Error: 0.0912
- R2 Score: 0.9838

Model performance for Test set
- Root Mean Squared Error: 0.2135
- Mean Absolute Error: 0.1696
- R2 Score: 0.9449
## ===================================


AdaBoost Regressor
Model performance for Training set
- Root Mean Squared Error: 0.2572
- Mean Absolute Error: 0.2094
- R2 Score: 0.9212

Model performance for Test set
- Root Mean Squared Error: 0.2625
- Mean Absolute Error: 0.2083
- R2 Score: 0.9167
## ===================================




# MODEL EVALUATION

In [ ]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)

 CHOOSING THE BEST MODEL - LINEAR REGRESSION?


In [ ]:
lin_model = LinearRegression(fit_intercept=True)
lin_model = lin_model.fit(X_train_scaled, y_train)
y_pred = lin_model.predict(X_test_scaled)
score = r2_score(y_test, y_pred)*100
print(" Accuracy of the model is %.2f" %score)

* Linear Regression outperforms other Regression Models, achieving a good R2 Score of 0.95.
* As an additional step, we can perform Hyper-parameter tuning. The individual model results may improve but the overall results may not increase much from this stage.

# PREDICTION ON NEW DATA INSTANCE 

In [ ]:
X_test_scaled[4]


In [ ]:
lin_model.predict([[ 1.37285117, -1.0291761 , -0.84616672, -1.78504629, -1.08250988,
        1.45955945, -0.65799605, -1.02021491, -0.78293347,  1.51039849,
       -0.49705784, -0.43041146]])

In [ ]:
y_test[4:5]
